In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [3]:
PROJECT_ROOT = Path.cwd().parent

data_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "brfss_2015"
    / "diabetes_binary_health_indicators_BRFSS2015.csv"
)

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (253680, 22)


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [4]:
target_column = "Diabetes_binary"

X = df.drop(columns=[target_column])
y = df[target_column]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(y.value_counts(normalize=True).mul(100).round(2))

X shape: (253680, 21)
y shape: (253680,)

Target distribution:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64

Target percentage:
Diabetes_binary
0.0    86.07
1.0    13.93
Name: proportion, dtype: float64


In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Temporary shape:", X_temp.shape)

Training shape: (177576, 21)
Temporary shape: (76104, 21)


In [6]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (177576, 21)
Validation: (38052, 21)
Test: (38052, 21)


In [7]:
split_distribution = pd.DataFrame({
    "Train": y_train.value_counts(normalize=True),
    "Validation": y_val.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True)
}) * 100

split_distribution.round(2)

,Train,Validation,Test
Diabetes_binary,,,
0.0,86.07,86.07,86.07
1.0,13.93,13.93,13.93


In [8]:
binary_features = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "HeartDiseaseorAttack",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex",
]

ordinal_features = [
    "GenHlth",
    "Age",
    "Education",
    "Income",
]

count_features = [
    "MentHlth",
    "PhysHlth",
]

continuous_features = [
    "BMI",
]

In [9]:
all_features = (
    binary_features
    + ordinal_features
    + count_features
    + continuous_features
)

print("Total defined features:", len(all_features))
print("Features in dataset:", len(X.columns))

print("Missing from definition:", set(X.columns) - set(all_features))
print("Extra in definition:", set(all_features) - set(X.columns))

Total defined features: 21
Features in dataset: 21
Missing from definition: set()
Extra in definition: set()


In [10]:
numeric_features = (
    ordinal_features
    + count_features
    + continuous_features
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "binary",
            "passthrough",
            binary_features
        ),
    ],
    remainder="drop"
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('binary', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature

In [11]:
preprocessor.fit_transform(X)

array([[ 2.32912057,  0.31690008, -1.06559465, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.45729435, -0.33793279,  0.96327159, ...,  1.        ,
         0.        ,  0.        ],
       [ 2.32912057,  0.31690008, -1.06559465, ...,  1.        ,
         1.        ,  0.        ],
       ...,
       [-1.41453187, -1.97501498, -0.05116153, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.45729435, -0.33793279, -0.05116153, ...,  0.        ,
         0.        ,  1.        ],
       [-0.47861876,  0.31690008,  0.96327159, ...,  0.        ,
         0.        ,  0.        ]], shape=(253680, 21))

In [12]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (177576, 21)
Processed validation shape: (38052, 21)
Processed test shape: (38052, 21)


In [13]:
print(
    "NaN values in training:",
    np.isnan(X_train_processed).sum()
)

print(
    "NaN values in validation:",
    np.isnan(X_val_processed).sum()
)

print(
    "NaN values in test:",
    np.isnan(X_test_processed).sum()
)

NaN values in training: 0
NaN values in validation: 0
NaN values in test: 0


In [14]:
processed_dir = PROJECT_ROOT / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(
    processed_dir / "X_train.csv",
    index=False
)

X_val.to_csv(
    processed_dir / "X_val.csv",
    index=False
)

X_test.to_csv(
    processed_dir / "X_test.csv",
    index=False
)

y_train.to_csv(
    processed_dir / "y_train.csv",
    index=False
)

y_val.to_csv(
    processed_dir / "y_val.csv",
    index=False
)

y_test.to_csv(
    processed_dir / "y_test.csv",
    index=False
)

print("Train, validation, and test splits saved successfully.")

Train, validation, and test splits saved successfully.
